# DSA 8301 - Statistical Inference for Big Data
## Kenya Housing Survey 2023/24: Housing Financial Vulnerability Score (HFVS)

**Student:** Valerie Jerono | **Reg. No.:** 222331  
**Institution:** Strathmore University iLabAfrica  
**Dataset:** Kenya Housing Survey (KHS) 2023/24, KNBS  
**Main output:** A <=50-column inference frame, a transparent five-pillar HFVS score, and report-ready statistical tests.

### Analytical story
Kenya's housing challenge is not only about whether a household has a roof. It is also about whether that household can afford the roof, occupy it securely, survive physical hazards, and access basic utilities. This notebook turns the cleaned KHS master frame into a compact, statistically defensible inference dataset and asks: **which household, dwelling, tenure, hazard, utility, and demographic factors are most strongly associated with housing financial vulnerability?**


## Notebook map and workflow alignment

This notebook follows the project workflow in sequence:

1. **Phase 1 - Load data:** load `master_frame_clean.parquet` from the Colab Drive project folder.
2. **Phase 2 - Reduce columns:** reduce the cleaned 431-column frame to a <=50-column inference frame using pillar coverage, validators, survey weights, and proxy/context variables.
3. **Phase 3 - EDA and normality:** describe variables, visualize distributions, compare groups, and run Shapiro-Wilk normality checks.
4. **Phase 4 - Parametric inference:** t-test, Welch t-test, ANOVA, confidence interval, and multiple regression.
5. **Phase 5 - Nonparametric inference:** Wilcoxon, Mann-Whitney U, Kruskal-Wallis, Spearman correlation, bootstrap intervals, and chi-square validation tests.
6. **Phase 6 - Report assets:** save figures, result tables, column-reduction audit, county atlas, and the reduced frame.

Important design rule from the business understanding document: the inference frame preserves the five HFVS pillars while avoiding an uncontrolled 400+ column analysis. The 50 columns are the **source variables**. Derived HFVS columns are created after reduction so the reduction remains auditable.


In [ ]:
# 0.1 Colab setup and dependencies
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception as exc:
    IN_COLAB = False
    print(f'Colab Drive mount skipped: {exc}')

!pip install -q pyarrow scipy statsmodels scikit-learn


In [ ]:
# 0.2 Imports and display settings
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 120)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'
RANDOM_STATE = 8301
rng = np.random.default_rng(RANDOM_STATE)

print('Environment ready.')


In [ ]:
# 0.3 Project paths
DRIVE_ROOT = Path('/content/drive/MyDrive/KHS_Dissertation')
BASE = DRIVE_ROOT if DRIVE_ROOT.exists() else Path.cwd()
PQ = BASE / 'data' / 'parquet'
FIGS = BASE / 'outputs' / 'figures' / 'dsa8301'
TABS = BASE / 'outputs' / 'tables' / 'dsa8301'

for folder in [PQ, FIGS, TABS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'BASE = {BASE}')
print(f'PQ   = {PQ}')
print(f'FIGS = {FIGS}')
print(f'TABS = {TABS}')


## 1. Load the cleaned master frame

The cleaning notebook saved `master_frame_clean.parquet` after sentinel decoding, structural-missingness treatment, tenure-stratified imputation, monetary winsorisation, consistency checks, and feature engineering. This notebook starts from that cleaned frame rather than the raw 443-column master.


In [ ]:
# 1.1 Load cleaned frame
candidates = [
    PQ / 'master_frame_clean.parquet',
    Path('master_frame_clean.parquet'),
    Path('/content/master_frame_clean.parquet'),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find master_frame_clean.parquet. Run DSA8301_Phase2_Cleaning_v2.ipynb first, '
        'or place the cleaned parquet in the project data/parquet folder.'
    )

df_raw = pd.read_parquet(DATA_PATH)
print(f'Loaded: {DATA_PATH}')
print(f'Raw cleaned frame: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')

display(df_raw.head(3))


## 2. Reduce 431 columns to 50 or fewer

The reduction is not arbitrary. It keeps:

- identifiers, geography, and survey weights needed for representative summaries;
- direct five-pillar HFVS ingredients;
- validation variables such as stated housing-cost burden and missed payment;
- a small set of demographic proxy/context variables for group comparisons and regression.

Variables that are highly redundant, free-text, administrative, or outside the household-level inference story remain in the cleaning notebook and source parquet, but are not carried into the main inference frame.


In [ ]:
# 2.1 A role-audited 50-column source-variable plan
COLUMN_PLAN = [
    # Identifiers, geography, survey design
    ('hh_id', 'identifier', 'Unique household interview key'),
    ('county_code', 'geography', 'County code, 1-47'),
    ('county_name', 'geography', 'County name for reporting'),
    ('urban_rural', 'geography', 'Residence type: 1=Urban, 2=Rural'),
    ('hh_weight', 'survey_design', 'Household survey weight'),

    # Pillar 1 - financial stress
    ('total_exp', 'pillar_1_financial', 'Total monthly expenditure proxy'),
    ('spend_housing_kes', 'pillar_1_financial', 'Monthly housing expenditure'),
    ('rent_actual_kes', 'pillar_1_financial', 'Monthly rent, renter subpopulation'),
    ('rent_burden_ratio', 'pillar_1_financial', 'Rent burden ratio from cleaning notebook'),
    ('rent_burdened', 'pillar_1_financial', 'Rent burden >30 percent threshold'),
    ('housing_cost_burden', 'validator', 'Self-reported housing cost burden'),
    ('missed_payment', 'validator', 'Missed housing payment in past 12 months'),
    ('mort_no_market', 'pillar_1_financial', 'County has no formal mortgage market flag'),

    # Pillar 2 - physical quality
    ('hh_size', 'pillar_2_quality', 'Household size'),
    ('dw_rooms', 'pillar_2_quality', 'Number of dwelling rooms'),
    ('dw_bedrooms', 'pillar_2_quality', 'Number of bedrooms'),
    ('persons_per_room', 'pillar_2_quality', 'Crowding index'),
    ('is_overcrowded', 'pillar_2_quality', 'WHO-style crowding flag, >2 persons per room'),
    ('dw_type', 'pillar_2_quality', 'Dwelling type'),
    ('obj_quality_score', 'pillar_2_quality', 'Objective material quality score'),
    ('perc_overall', 'pillar_2_quality', 'Subjective overall housing perception'),

    # Pillar 3 - tenure security
    ('is_renter', 'pillar_3_tenure', 'Renter flag'),
    ('is_owner', 'pillar_3_tenure', 'Owner flag'),
    ('tenure_type', 'pillar_3_tenure', 'Tenure type'),
    ('has_title_doc', 'pillar_3_tenure', 'Has formal title or ownership document'),
    ('eviction_risk', 'pillar_3_tenure', 'Eviction risk'),
    ('owns_land', 'pillar_3_tenure', 'Owns any land parcel'),
    ('lp_n_parcels', 'pillar_3_tenure', 'Number of land parcels'),
    ('lp_has_title', 'pillar_3_tenure', 'Land parcel has title'),
    ('satisfied_tenure', 'pillar_3_tenure', 'Satisfied with tenure situation'),
    ('yrs_in_dwelling', 'pillar_3_tenure', 'Years or occupancy year in current dwelling'),

    # Pillar 4 - hazard exposure
    ('flood_exposure', 'pillar_4_hazard', 'Flood exposure'),
    ('landslide_exposure', 'pillar_4_hazard', 'Landslide exposure'),
    ('other_hazard_exposure', 'pillar_4_hazard', 'Other environmental hazard exposure'),
    ('dw_in_hazard_zone', 'pillar_4_hazard', 'Dwelling located in hazard-prone zone'),
    ('triple_exposed', 'pillar_4_hazard', 'Flood/informal/no-title compound risk flag'),

    # Pillar 5 - utility deprivation
    ('water_src_main', 'pillar_5_utility', 'Primary water source'),
    ('water_treated', 'pillar_5_utility', 'Water treatment flag'),
    ('water_dist_mins', 'pillar_5_utility', 'Minutes to water source'),
    ('toilet_type', 'pillar_5_utility', 'Toilet type'),
    ('has_handwash_facility', 'pillar_5_utility', 'Handwashing facility flag'),
    ('lighting_src', 'pillar_5_utility', 'Main lighting source'),
    ('cooking_fuel', 'pillar_5_utility', 'Main cooking fuel'),
    ('util_burden_ratio', 'pillar_5_utility', 'Utility burden ratio'),

    # Proxy/context variables for group comparisons and regression
    ('dependency_ratio', 'proxy_context', 'Household dependency ratio'),
    ('mean_age', 'proxy_context', 'Mean household age from roster'),
    ('max_edu_isced', 'proxy_context', 'Maximum education level in household'),
    ('hh_head_sex', 'proxy_context', 'Sex of household head'),
    ('has_disability', 'proxy_context', 'Any household member has disability'),
    ('owns_mobile', 'proxy_context', 'Mobile phone ownership proxy'),
]

assert len(COLUMN_PLAN) == 50, f'Plan should contain exactly 50 source variables, found {len(COLUMN_PLAN)}.'

planned_cols = [c for c, _, _ in COLUMN_PLAN]
present_cols = [c for c in planned_cols if c in df_raw.columns]
missing_cols = [c for c in planned_cols if c not in df_raw.columns]

analysis_50 = df_raw[present_cols].copy()
assert analysis_50.shape[1] <= 50

reduction_audit = pd.DataFrame(COLUMN_PLAN, columns=['column', 'role', 'reason'])
reduction_audit['present'] = reduction_audit['column'].isin(present_cols)
reduction_audit['missing_pct_in_source'] = [
    df_raw[c].isna().mean() * 100 if c in df_raw.columns else np.nan
    for c in reduction_audit['column']
]
reduction_audit['dtype_in_source'] = [
    str(df_raw[c].dtype) if c in df_raw.columns else 'missing'
    for c in reduction_audit['column']
]

print(f'Source columns before reduction : {df_raw.shape[1]:,}')
print(f'Source columns after reduction  : {analysis_50.shape[1]:,}')
print(f'Missing planned columns         : {len(missing_cols)}')
if missing_cols:
    print(missing_cols)

reduction_audit.to_csv(TABS / 'statistical_inference_column_reduction_audit.csv', index=False)
analysis_50.to_parquet(PQ / 'statistical_inference_analysis_frame_50cols.parquet', index=False)
print(f'Saved <=50-column frame -> {PQ / "statistical_inference_analysis_frame_50cols.parquet"}')

display(reduction_audit)


In [ ]:
# 2.2 Role balance and missingness after reduction
role_summary = (
    reduction_audit.query('present')
    .groupby('role', as_index=False)
    .agg(n_columns=('column', 'count'), mean_missing_pct=('missing_pct_in_source', 'mean'))
    .sort_values('role')
)

missing_after_reduction = (
    analysis_50.isna().mean().mul(100).rename('missing_pct')
    .reset_index().rename(columns={'index': 'column'})
    .merge(reduction_audit[['column', 'role', 'reason']], on='column', how='left')
    .sort_values('missing_pct', ascending=False)
)

print('Role balance in <=50-column frame:')
display(role_summary)

print('Highest missingness columns retained, usually structural by design:')
display(missing_after_reduction.head(15))


## 3. Data quality checks before inference

The cleaning notebook already handled the major issues. This section adds inference-specific guards:

- impossible ages are removed from the analytical `mean_age_clean` field;
- `yrs_in_dwelling` is checked because the output suggests it behaves like a year of occupancy, not elapsed years;
- rent burden is recomputed for the renter subpopulation so the validation tests do not depend on a possibly sparse stored ratio;
- survey weights are cleaned for weighted county summaries.


In [ ]:
# 3.1 Helper functions and inference-specific cleaned fields
analysis = analysis_50.copy()


def num(col, frame=analysis):
    if col not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[col], errors='coerce')


def binary01(col, frame=analysis):
    s = num(col, frame)
    return s.where(s.isin([0, 1]))

# Labels for readable figures
analysis['residence_label'] = num('urban_rural').map({1: 'Urban', 2: 'Rural'}).fillna('Unknown')

# Survey weights: keep only positive finite weights; fall back to 1 if absent.
weights = num('hh_weight')
analysis['hh_weight_clean'] = weights.where(np.isfinite(weights) & (weights > 0), 1.0).fillna(1.0)

# Mean age sanity check: values above 110 are not plausible household mean ages.
age = num('mean_age')
analysis['mean_age_clean'] = age.where(age.between(0, 110))

# Tenure duration sanity check: the cleaning output suggests yrs_in_dwelling may behave like
# a calendar year for many rows. If the median is above 150, interpret as occupancy year.
yrs_raw = num('yrs_in_dwelling')
if yrs_raw.notna().sum() and yrs_raw.median() > 150:
    tenure_years = 2024 - yrs_raw
    tenure_interpretation = 'interpreted as occupancy year and converted to duration'
else:
    tenure_years = yrs_raw
    tenure_interpretation = 'interpreted as elapsed years'
analysis['tenure_years_clean'] = tenure_years.where(tenure_years.between(0, 100))

if 'satisfied_tenure' in analysis.columns:
    analysis['satisfied_tenure_clean'] = binary01('satisfied_tenure')
else:
    analysis['satisfied_tenure_clean'] = np.nan

# Recompute rent burden for renters using the documented formula.
is_renter = binary01('is_renter').eq(1)
rent = num('rent_actual_kes')
total_exp = num('total_exp')
housing_exp = num('spend_housing_kes')
denominator = (total_exp - housing_exp).where((total_exp - housing_exp) > 0)
recomputed_ratio = (rent / denominator).where(is_renter & rent.gt(0) & denominator.gt(0))
recomputed_ratio = recomputed_ratio.clip(lower=0, upper=1.5)

stored_ratio = num('rent_burden_ratio')
analysis['rent_burden_ratio_model'] = recomputed_ratio.combine_first(stored_ratio)
analysis['rent_burdened_model'] = (analysis['rent_burden_ratio_model'] > 0.30).astype(float)
analysis.loc[~is_renter | analysis['rent_burden_ratio_model'].isna(), 'rent_burdened_model'] = np.nan

quality_notes = pd.DataFrame({
    'check': [
        'mean_age impossible values removed',
        'rent_burden_ratio non-null before recompute',
        'rent_burden_ratio non-null after recompute',
        'tenure years interpretation',
        'tenure duration impossible values removed',
        'renters in reduced frame',
        'positive survey weights',
    ],
    'value': [
        int(age.notna().sum() - analysis['mean_age_clean'].notna().sum()),
        int(stored_ratio.notna().sum()),
        int(analysis['rent_burden_ratio_model'].notna().sum()),
        tenure_interpretation,
        int(yrs_raw.notna().sum() - analysis['tenure_years_clean'].notna().sum()),
        int(is_renter.sum()),
        int(analysis['hh_weight_clean'].gt(0).sum()),
    ]
})

quality_notes.to_csv(TABS / 'statistical_inference_quality_notes.csv', index=False)
display(quality_notes)


## 4. Construct an interpretable HFVS score

This notebook needs one outcome for inference. The score below is a transparent teaching-score version of the five-pillar HFVS. It is designed for statistical inference and self-discovery, not as a black-box official index.

Scoring convention: **0 = less vulnerable, 1 = more vulnerable**. Each pillar is the mean of available risk components. The final `hfvs_score` is the mean of the five pillar scores.


In [ ]:
# 4.1 Risk-score helpers

def norm01(series, invert=False, q_low=0.01, q_high=0.99):
    s = pd.to_numeric(series, errors='coerce').astype(float)
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)
    lo, hi = s.quantile(q_low), s.quantile(q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        out = pd.Series(np.nan, index=s.index)
    else:
        out = ((s.clip(lo, hi) - lo) / (hi - lo)).clip(0, 1)
    return 1 - out if invert else out


def risk_if_in(col, high_risk_values):
    s = num(col)
    return pd.Series(np.where(s.isin(high_risk_values), 1.0, 0.0), index=analysis.index).where(s.notna())


def risk_if_not_in(col, low_risk_values):
    s = num(col)
    return pd.Series(np.where(s.isin(low_risk_values), 0.0, 1.0), index=analysis.index).where(s.notna())


def risk_binary_yes(col):
    return binary01(col).astype(float)


def risk_binary_no(col):
    s = binary01(col)
    return (1 - s).where(s.notna())


def component_mean(name, components):
    comp = pd.DataFrame({k: v for k, v in components.items() if v is not None})
    comp = comp.dropna(axis=1, how='all')
    if comp.empty:
        analysis[name] = np.nan
        return comp
    analysis[name] = comp.mean(axis=1, skipna=True)
    coverage = comp.notna().mean().rename('coverage_rate').reset_index().rename(columns={'index': 'component'})
    coverage.insert(0, 'pillar', name)
    return coverage

coverage_tables = []

# Pillar 1: financial stress
rent_risk = (num('rent_burden_ratio_model') / 1.5).clip(0, 1)
coverage_tables.append(component_mean('p1_financial_stress', {
    'low_expenditure_risk': norm01(np.log1p(num('total_exp')), invert=True),
    'rent_burden_risk': rent_risk,
    'stated_cost_burden': risk_binary_yes('housing_cost_burden'),
    'missed_payment': risk_binary_yes('missed_payment'),
    'mortgage_market_absent': risk_binary_yes('mort_no_market'),
}))

# Pillar 2: physical quality
perc = num('perc_overall')
subjective_risk = ((perc - 1) / 2).clip(0, 1).where(perc.notna())
coverage_tables.append(component_mean('p2_physical_quality', {
    'poor_objective_quality': (1 - num('obj_quality_score').clip(0, 1)).where(num('obj_quality_score').notna()),
    'crowding_continuous': norm01(num('persons_per_room')),
    'overcrowded_flag': risk_binary_yes('is_overcrowded'),
    'subjective_poor_overall': subjective_risk,
    'dwelling_type_risk_heuristic': norm01(num('dw_type')),
}))

# Pillar 3: tenure security
coverage_tables.append(component_mean('p3_tenure_security', {
    'renter_status': risk_binary_yes('is_renter'),
    'not_owner': risk_binary_no('is_owner'),
    'no_title_doc': risk_binary_no('has_title_doc'),
    'eviction_risk': risk_binary_yes('eviction_risk'),
    'landless': risk_binary_no('owns_land'),
    'land_without_title': risk_binary_no('lp_has_title'),
    'not_satisfied_tenure': risk_binary_no('satisfied_tenure_clean'),
    'short_tenure_duration': norm01(analysis['tenure_years_clean'], invert=True),
}))

# Pillar 4: hazard exposure
coverage_tables.append(component_mean('p4_hazard_exposure', {
    'flood_exposure': norm01(num('flood_exposure')),
    'landslide_exposure': norm01(num('landslide_exposure')),
    'other_hazard_exposure': norm01(num('other_hazard_exposure')),
    'dwelling_hazard_zone': risk_binary_yes('dw_in_hazard_zone'),
    'triple_exposure': risk_binary_yes('triple_exposed'),
}))

# Pillar 5: utility deprivation
# Codebook hypotheses used here: improved water is usually coded as low values; unsafe sanitation often includes 7/8;
# firewood/charcoal are 7/9 in the project notes; grid/solar lighting are treated as lower risk.
coverage_tables.append(component_mean('p5_utility_deprivation', {
    'unimproved_water_source': risk_if_not_in('water_src_main', [1, 2, 3]),
    'water_not_treated': risk_binary_no('water_treated'),
    'long_water_distance': norm01(num('water_dist_mins')),
    'unsafe_toilet_type': risk_if_in('toilet_type', [7, 8]),
    'no_handwash_facility': risk_binary_no('has_handwash_facility'),
    'risky_lighting_source': risk_if_not_in('lighting_src', [1, 2, 3, 4]),
    'solid_cooking_fuel': risk_if_in('cooking_fuel', [7, 9]),
    'utility_burden': num('util_burden_ratio').clip(0, 1),
}))

pillar_cols = [
    'p1_financial_stress',
    'p2_physical_quality',
    'p3_tenure_security',
    'p4_hazard_exposure',
    'p5_utility_deprivation',
]
analysis['hfvs_score'] = analysis[pillar_cols].mean(axis=1, skipna=True)
analysis['hfvs_class'] = pd.cut(
    analysis['hfvs_score'],
    bins=[-0.01, 0.20, 0.40, 0.60, 0.80, 1.00],
    labels=['Very low', 'Low', 'Moderate', 'High', 'Very high']
)
hfvs_qcut = pd.qcut(analysis['hfvs_score'], q=5, duplicates='drop')
analysis['hfvs_quintile'] = hfvs_qcut.astype(str).replace({'nan': np.nan})

component_coverage = pd.concat(coverage_tables, ignore_index=True)
component_coverage.to_csv(TABS / 'statistical_inference_pillar_component_coverage.csv', index=False)

print('HFVS score constructed.')
print(analysis[pillar_cols + ['hfvs_score']].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])
display(component_coverage)


In [ ]:
# 4.2 Quick HFVS profile
hfvs_profile = pd.DataFrame({
    'metric': [
        'N households scored', 'Mean HFVS', 'Median HFVS', 'Std HFVS',
        'Share high or very high', 'Share triple exposed', 'Share overcrowded'
    ],
    'value': [
        analysis['hfvs_score'].notna().sum(),
        analysis['hfvs_score'].mean(),
        analysis['hfvs_score'].median(),
        analysis['hfvs_score'].std(),
        analysis['hfvs_class'].isin(['High', 'Very high']).mean(),
        binary01('triple_exposed').mean(),
        binary01('is_overcrowded').mean(),
    ]
})

hfvs_profile.to_csv(TABS / 'statistical_inference_hfvs_profile.csv', index=False)
display(hfvs_profile)

# PyArrow cannot always write pandas Interval-backed columns, especially qcut bins.
# Save a Parquet-safe copy while keeping the in-memory analysis frame unchanged.
parquet_safe = analysis.copy()
for c in parquet_safe.columns:
    if isinstance(parquet_safe[c].dtype, (pd.CategoricalDtype, pd.IntervalDtype)):
        parquet_safe[c] = parquet_safe[c].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

parquet_safe.to_parquet(PQ / 'statistical_inference_scored_frame.parquet', index=False)
print(f'Scored analysis frame saved -> {PQ / "statistical_inference_scored_frame.parquet"}')


## 5. Exploratory data analysis

EDA answers three practical questions before hypothesis testing:

1. What does vulnerability look like overall?
2. Which pillars appear to drive the score?
3. Do key relationships differ by urban/rural residence, tenure, hazard exposure, or deprivation?


In [ ]:
# 5.1 Descriptive statistics table
key_numeric = [
    'hfvs_score',
    'p1_financial_stress', 'p2_physical_quality', 'p3_tenure_security',
    'p4_hazard_exposure', 'p5_utility_deprivation',
    'total_exp', 'rent_burden_ratio_model', 'persons_per_room',
    'obj_quality_score', 'util_burden_ratio', 'dependency_ratio',
    'mean_age_clean', 'hh_size', 'tenure_years_clean'
]
key_numeric = [c for c in key_numeric if c in analysis.columns]

def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

def span(x):
    return x.max() - x.min()

desc = analysis[key_numeric].agg(['count', 'mean', 'median', 'std', 'var', span, iqr]).T
desc.columns = ['N', 'Mean', 'Median', 'Std Dev', 'Variance', 'Range', 'IQR']
desc.to_csv(TABS / 'statistical_inference_descriptive_statistics.csv')

display(desc)


In [ ]:
# 5.2 Weighted and unweighted HFVS by residence

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors='coerce')
    weights = pd.to_numeric(weights, errors='coerce')
    mask = values.notna() & weights.notna() & (weights > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(values[mask], weights=weights[mask])

residence_summary = []
for label, g in analysis.groupby('residence_label', dropna=False):
    residence_summary.append({
        'residence': label,
        'n': len(g),
        'mean_hfvs_unweighted': g['hfvs_score'].mean(),
        'mean_hfvs_weighted': weighted_mean(g['hfvs_score'], g['hh_weight_clean']),
        'median_total_exp': g['total_exp'].median(),
        'pct_overcrowded': binary01('is_overcrowded', g).mean(),
        'pct_triple_exposed': binary01('triple_exposed', g).mean(),
    })
residence_summary = pd.DataFrame(residence_summary)
residence_summary.to_csv(TABS / 'statistical_inference_residence_summary.csv', index=False)

display(residence_summary)


In [ ]:
# 5.3 Distribution plots for HFVS and pillars
plot_cols = ['hfvs_score'] + pillar_cols
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, plot_cols):
    sns.histplot(analysis[col].dropna(), kde=True, ax=ax, color='#2A7F62')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Score: 0 less vulnerable, 1 more vulnerable')

for ax in axes[len(plot_cols):]:
    ax.axis('off')

fig.suptitle('HFVS and Five Pillar Distributions', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_hfvs_pillar_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 5.4 Relationship heatmap and key scatter plot
corr_cols = [
    'hfvs_score', 'p1_financial_stress', 'p2_physical_quality', 'p3_tenure_security',
    'p4_hazard_exposure', 'p5_utility_deprivation', 'total_exp', 'persons_per_room',
    'obj_quality_score', 'util_burden_ratio', 'dependency_ratio', 'mean_age_clean'
]
corr_cols = [c for c in corr_cols if c in analysis.columns]

corr = analysis[corr_cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set_title('Spearman Correlation Heatmap: HFVS, Pillars, and Core Drivers')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_spearman_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

sample = analysis.dropna(subset=['total_exp', 'hfvs_score']).sample(
    n=min(4000, analysis.dropna(subset=['total_exp', 'hfvs_score']).shape[0]),
    random_state=RANDOM_STATE
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=sample, x='total_exp', y='hfvs_score', hue='residence_label', alpha=0.45, s=25, ax=ax)
ax.set_xscale('log')
ax.set_title('HFVS vs Total Monthly Expenditure')
ax.set_xlabel('Total expenditure, log scale (KES/month)')
ax.set_ylabel('HFVS score')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_hfvs_vs_expenditure.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 5.5 Objective-subjective quality gap
# perc_overall is interpreted as 1=good, 2=fair, 3=poor based on the project reference notes.
perc = num('perc_overall')
analysis['subjective_quality_risk'] = ((perc - 1) / 2).clip(0, 1).where(perc.notna())
analysis['objective_quality_risk'] = 1 - num('obj_quality_score').clip(0, 1)
analysis['quality_gap'] = analysis['objective_quality_risk'] - analysis['subjective_quality_risk']
analysis['latent_quality_risk'] = (analysis['quality_gap'] > 0.30).astype(float).where(analysis['quality_gap'].notna())

gap_summary = analysis[['objective_quality_risk', 'subjective_quality_risk', 'quality_gap', 'latent_quality_risk']].describe().T
print('Quality gap: positive values mean objective conditions are worse than self-perception suggests.')
display(gap_summary)

county_gap = (
    analysis.groupby(['county_code', 'county_name'], dropna=False)
    .agg(n=('hfvs_score', 'size'), mean_quality_gap=('quality_gap', 'mean'), pct_latent_risk=('latent_quality_risk', 'mean'))
    .reset_index()
    .query('n >= 50')
    .sort_values('pct_latent_risk', ascending=False)
)
county_gap.to_csv(TABS / 'statistical_inference_quality_gap_by_county.csv', index=False)

print('Top counties by latent objective quality risk:')
display(county_gap.head(12))


## 6. Normality decision gate

The workflow requires normality checks before choosing tests. With more than 21,000 observations, Shapiro-Wilk can reject even tiny deviations, so the table includes skewness and kurtosis as practical context. The decision rule remains: **Shapiro p > 0.05 suggests approximate normality; p <= 0.05 points us toward nonparametric alternatives.**


In [ ]:
# 6.1 Shapiro-Wilk normality assessment
normality_cols = [
    'hfvs_score', 'p1_financial_stress', 'p2_physical_quality', 'p3_tenure_security',
    'p4_hazard_exposure', 'p5_utility_deprivation', 'total_exp', 'persons_per_room',
    'obj_quality_score', 'util_burden_ratio', 'dependency_ratio', 'mean_age_clean'
]
normality_cols = [c for c in normality_cols if c in analysis.columns]

normality_rows = []
for col in normality_cols:
    x = pd.to_numeric(analysis[col], errors='coerce').dropna()
    if len(x) < 3:
        continue
    sample = x.sample(n=min(5000, len(x)), random_state=RANDOM_STATE)
    stat, p = stats.shapiro(sample)
    normality_rows.append({
        'variable': col,
        'n_nonmissing': len(x),
        'sample_used': len(sample),
        'shapiro_W': stat,
        'p_value': p,
        'skew': stats.skew(x, nan_policy='omit'),
        'kurtosis': stats.kurtosis(x, nan_policy='omit'),
        'normal_at_0_05': 'Yes' if p > 0.05 else 'No',
        'recommended_family': 'Parametric acceptable' if p > 0.05 else 'Use nonparametric / robust checks',
    })

normality = pd.DataFrame(normality_rows).sort_values('p_value')
normality.to_csv(TABS / 'statistical_inference_normality_results.csv', index=False)
display(normality)


In [ ]:
# 6.2 Q-Q plots for core variables
qq_cols = ['hfvs_score', 'total_exp', 'persons_per_room', 'obj_quality_score', 'util_burden_ratio', 'dependency_ratio']
qq_cols = [c for c in qq_cols if c in analysis.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for ax, col in zip(axes, qq_cols):
    x = pd.to_numeric(analysis[col], errors='coerce').dropna()
    if len(x) > 5000:
        x = x.sample(5000, random_state=RANDOM_STATE)
    stats.probplot(x, dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: {col}')

for ax in axes[len(qq_cols):]:
    ax.axis('off')

fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_qq_plots.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Hypothesis-testing helpers

Every test is stored in one results table using the report structure required by the workflow: research question, hypotheses, assumptions, statistic, degrees of freedom, p-value, decision, and plain-English interpretation.


In [ ]:
# 7.1 Shared test reporting functions
ALPHA = 0.05
TEST_RESULTS = []


def p_to_decision(p, alpha=ALPHA):
    if pd.isna(p):
        return 'Estimate only'
    return 'Reject H0' if p < alpha else 'Fail to reject H0'


def add_result(test, variables, question, h0, h1, assumptions, statistic, df, p_value, interpretation):
    TEST_RESULTS.append({
        'test': test,
        'variables': variables,
        'research_question': question,
        'H0': h0,
        'H1': h1,
        'assumptions_checked': assumptions,
        'statistic': statistic,
        'df': df,
        'p_value': p_value,
        'decision_alpha_0_05': p_to_decision(p_value),
        'plain_english_interpretation': interpretation,
    })


def cohen_d(a, b):
    a = pd.Series(a).dropna()
    b = pd.Series(b).dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else np.nan


def welch_df(a, b):
    a = pd.Series(a).dropna()
    b = pd.Series(b).dropna()
    va, vb = a.var(ddof=1), b.var(ddof=1)
    na, nb = len(a), len(b)
    nume = (va / na + vb / nb) ** 2
    deno = ((va / na) ** 2 / (na - 1)) + ((vb / nb) ** 2 / (nb - 1))
    return nume / deno if deno > 0 else np.nan


def cramers_v(table):
    chi2, _, _, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1))) if n > 0 else np.nan

print('Test helpers ready.')


## 8. Parametric statistical analysis

Parametric methods are included because they are required by the workflow and remain useful with large survey data when paired with diagnostics and robust interpretation. Where normality is questionable, the next section repeats the core questions with nonparametric alternatives.


In [ ]:
# 8.1 One-sample t-test and confidence interval for mean HFVS
x = analysis['hfvs_score'].dropna()
benchmark = 0.50

t_stat, t_p = stats.ttest_1samp(x, popmean=benchmark)
ci = stats.t.interval(0.95, df=len(x) - 1, loc=x.mean(), scale=stats.sem(x))

add_result(
    test='One-sample t-test',
    variables='hfvs_score vs 0.50 benchmark',
    question='Is average housing financial vulnerability different from the moderate-risk benchmark of 0.50?',
    h0='Mean HFVS = 0.50',
    h1='Mean HFVS != 0.50',
    assumptions='Continuous score, independent households, Q-Q/normality checked; large n supports t approximation.',
    statistic=f't = {t_stat:.4f}',
    df=f'{len(x) - 1}',
    p_value=t_p,
    interpretation=f'Mean HFVS is {x.mean():.3f}; 95% CI is ({ci[0]:.3f}, {ci[1]:.3f}).'
)

add_result(
    test='t-based confidence interval',
    variables='hfvs_score',
    question='What is the plausible population range for mean HFVS?',
    h0='Not a null-hypothesis test',
    h1='Not applicable',
    assumptions='Continuous score, independent households, standard error estimated from sample.',
    statistic=f'mean = {x.mean():.4f}',
    df=f'{len(x) - 1}',
    p_value=np.nan,
    interpretation=f'The 95% t interval for mean HFVS is ({ci[0]:.3f}, {ci[1]:.3f}).'
)

print(f'One-sample t-test: t={t_stat:.4f}, p={t_p:.6g}, 95% CI=({ci[0]:.4f}, {ci[1]:.4f})')


In [ ]:
# 8.2 Welch independent t-test: Urban vs Rural HFVS
urban = analysis.loc[analysis['residence_label'].eq('Urban'), 'hfvs_score'].dropna()
rural = analysis.loc[analysis['residence_label'].eq('Rural'), 'hfvs_score'].dropna()

lev_stat, lev_p = stats.levene(urban, rural)
t_stat, t_p = stats.ttest_ind(urban, rural, equal_var=False)
df_welch = welch_df(urban, rural)
d = cohen_d(urban, rural)

add_result(
    test='Welch independent t-test',
    variables='hfvs_score by urban_rural',
    question='Does mean HFVS differ between urban and rural households?',
    h0='Mean HFVS urban = Mean HFVS rural',
    h1='Mean HFVS urban != Mean HFVS rural',
    assumptions=f'Independent groups; Levene p={lev_p:.4g}; Welch version used for unequal variances.',
    statistic=f't = {t_stat:.4f}; Cohen d = {d:.4f}',
    df=f'{df_welch:.1f}',
    p_value=t_p,
    interpretation=f'Urban mean={urban.mean():.3f}, rural mean={rural.mean():.3f}; positive d means urban is higher.'
)

print(f'Welch t-test: t={t_stat:.4f}, df={df_welch:.1f}, p={t_p:.6g}, Cohen d={d:.3f}')


In [ ]:
# 8.3 One-way ANOVA: HFVS across dependency-ratio bands
analysis['dependency_ratio_clean'] = num('dependency_ratio').where(num('dependency_ratio').between(0, 10))
analysis['dependency_band'] = pd.cut(
    analysis['dependency_ratio_clean'],
    bins=[-np.inf, 0.5, 1.0, np.inf],
    labels=['Low dependency', 'Moderate dependency', 'High dependency']
)

groups = [g['hfvs_score'].dropna() for _, g in analysis.groupby('dependency_band', observed=True)]
group_names = [str(k) for k, _ in analysis.groupby('dependency_band', observed=True)]
f_stat, f_p = stats.f_oneway(*groups)

add_result(
    test='One-way ANOVA',
    variables='hfvs_score by dependency_band',
    question='Does mean HFVS differ across household dependency levels?',
    h0='All dependency-band mean HFVS values are equal',
    h1='At least one dependency-band mean HFVS differs',
    assumptions='Independent groups; normality checked; homogeneity inspected with nonparametric follow-up.',
    statistic=f'F = {f_stat:.4f}',
    df=f'{len(groups) - 1}, {sum(len(g) for g in groups) - len(groups)}',
    p_value=f_p,
    interpretation='ANOVA compares whether vulnerability changes across low, moderate, and high dependency households.'
)

anova_summary = analysis.groupby('dependency_band', observed=True)['hfvs_score'].agg(['count', 'mean', 'median', 'std'])
display(anova_summary)
print(f'ANOVA: F={f_stat:.4f}, p={f_p:.6g}')

if f_p < ALPHA:
    tukey_data = analysis[['hfvs_score', 'dependency_band']].dropna()
    tukey = pairwise_tukeyhsd(tukey_data['hfvs_score'], tukey_data['dependency_band'])
    print(tukey.summary())


In [ ]:
# 8.4 Multiple linear regression with robust standard errors
reg_cols = ['hfvs_score', 'urban_rural', 'dependency_ratio_clean', 'hh_size', 'max_edu_isced', 'has_disability', 'owns_mobile', 'mean_age_clean']
reg_cols = [c for c in reg_cols if c in analysis.columns]
reg_df = analysis[reg_cols].copy()
for c in reg_cols:
    if c != 'hfvs_score':
        reg_df[c] = pd.to_numeric(reg_df[c], errors='coerce')
reg_df = reg_df.dropna()

formula = 'hfvs_score ~ C(urban_rural) + dependency_ratio_clean + hh_size + max_edu_isced + C(has_disability) + C(owns_mobile) + mean_age_clean'
model = smf.ols(formula, data=reg_df).fit(cov_type='HC3')
print(model.summary())

coef_table = model.summary2().tables[1].reset_index().rename(columns={'index': 'term'})
coef_table.to_csv(TABS / 'statistical_inference_regression_coefficients.csv', index=False)

add_result(
    test='Multiple linear regression',
    variables='hfvs_score ~ residence + dependency + household composition + education + assets',
    question='Which proxy/context variables explain variation in HFVS after adjusting for each other?',
    h0='All non-intercept regression coefficients jointly equal zero',
    h1='At least one predictor has a non-zero adjusted association with HFVS',
    assumptions='Linearity inspected; independent households; HC3 robust standard errors used for heteroscedasticity.',
    statistic=f'F = {model.fvalue:.4f}; R2 = {model.rsquared:.4f}',
    df=f'{int(model.df_model)}, {int(model.df_resid)}',
    p_value=model.f_pvalue,
    interpretation='Regression coefficients show adjusted association, not causal effects.'
)

# Residual diagnostics
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sm.qqplot(model.resid, line='s', ax=axes[0])
axes[0].set_title('Regression Residual Q-Q Plot')
axes[1].scatter(model.fittedvalues, model.resid, alpha=0.25, s=10)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Residuals vs Fitted Values')
axes[1].set_xlabel('Fitted HFVS')
axes[1].set_ylabel('Residual')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_regression_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Nonparametric statistical analysis

Most KHS variables are skewed, bounded, ordinal, or structurally missing. The nonparametric tests below are therefore not decorative; they are the more defensible family for many of the substantive questions.


In [ ]:
# 9.1 Wilcoxon signed-rank: HFVS against 0.50 benchmark
x = analysis['hfvs_score'].dropna()
wil_stat, wil_p = stats.wilcoxon(x - 0.50, zero_method='wilcox', alternative='two-sided')

add_result(
    test='Wilcoxon signed-rank',
    variables='hfvs_score vs 0.50 benchmark',
    question='Is the median HFVS different from the moderate-risk benchmark of 0.50?',
    h0='Median HFVS = 0.50',
    h1='Median HFVS != 0.50',
    assumptions='Nonparametric one-sample test; appropriate because HFVS is bounded and normality is unlikely.',
    statistic=f'W = {wil_stat:.4f}',
    df='-',
    p_value=wil_p,
    interpretation=f'Median HFVS is {x.median():.3f}; Wilcoxon tests whether it differs from 0.50.'
)

print(f'Wilcoxon: W={wil_stat:.4f}, p={wil_p:.6g}')


In [ ]:
# 9.2 Mann-Whitney U: Urban vs Rural HFVS
urban = analysis.loc[analysis['residence_label'].eq('Urban'), 'hfvs_score'].dropna()
rural = analysis.loc[analysis['residence_label'].eq('Rural'), 'hfvs_score'].dropna()
u_stat, u_p = stats.mannwhitneyu(urban, rural, alternative='two-sided')

add_result(
    test='Mann-Whitney U',
    variables='hfvs_score by urban_rural',
    question='Do urban and rural households differ in HFVS distribution?',
    h0='Urban and rural HFVS distributions are equal',
    h1='Urban and rural HFVS distributions differ',
    assumptions='Independent groups; nonparametric alternative to the two-sample t-test.',
    statistic=f'U = {u_stat:.4f}',
    df='-',
    p_value=u_p,
    interpretation=f'Urban median={urban.median():.3f}, rural median={rural.median():.3f}.'
)

print(f'Mann-Whitney U={u_stat:.4f}, p={u_p:.6g}')


In [ ]:
# 9.3 Kruskal-Wallis: HFVS across dependency bands
kw_groups = [g['hfvs_score'].dropna() for _, g in analysis.groupby('dependency_band', observed=True)]
kw_stat, kw_p = stats.kruskal(*kw_groups)

add_result(
    test='Kruskal-Wallis',
    variables='hfvs_score by dependency_band',
    question='Do dependency bands differ in HFVS distribution?',
    h0='All dependency-band HFVS distributions are equal',
    h1='At least one dependency-band distribution differs',
    assumptions='Independent groups; nonparametric alternative to one-way ANOVA.',
    statistic=f'H = {kw_stat:.4f}',
    df=f'{len(kw_groups) - 1}',
    p_value=kw_p,
    interpretation='Kruskal-Wallis tests median/rank differences across dependency groups.'
)

print(f'Kruskal-Wallis H={kw_stat:.4f}, p={kw_p:.6g}')


In [ ]:
# 9.4 Spearman rank correlations with HFVS
spearman_pairs = ['total_exp', 'persons_per_room', 'obj_quality_score', 'util_burden_ratio', 'dependency_ratio_clean', 'mean_age_clean']
for col in spearman_pairs:
    if col not in analysis.columns:
        continue
    sub = analysis[['hfvs_score', col]].dropna()
    if len(sub) < 5:
        continue
    rho, p = stats.spearmanr(sub['hfvs_score'], sub[col])
    add_result(
        test='Spearman rank correlation',
        variables=f'hfvs_score vs {col}',
        question=f'Is HFVS monotonically associated with {col}?',
        h0=f'Spearman rho between HFVS and {col} = 0',
        h1=f'Spearman rho between HFVS and {col} != 0',
        assumptions='Nonparametric monotonic association; robust to skew and ordinal-like variables.',
        statistic=f'rho = {rho:.4f}',
        df='-',
        p_value=p,
        interpretation=f'Rho={rho:.3f}; sign shows direction and magnitude shows rank association strength.'
    )

spearman_summary = pd.DataFrame([r for r in TEST_RESULTS if r['test'] == 'Spearman rank correlation'])
display(spearman_summary[['variables', 'statistic', 'p_value', 'decision_alpha_0_05', 'plain_english_interpretation']])


In [ ]:
# 9.5 Bootstrap confidence interval for median HFVS
x = analysis['hfvs_score'].dropna().to_numpy()
n_boot = 5000
boot_medians = np.empty(n_boot)
for i in range(n_boot):
    boot_medians[i] = np.median(rng.choice(x, size=len(x), replace=True))

boot_ci = np.percentile(boot_medians, [2.5, 97.5])

add_result(
    test='Bootstrap confidence interval',
    variables='median hfvs_score',
    question='What is the uncertainty interval for median HFVS without assuming normality?',
    h0='Not a null-hypothesis test',
    h1='Not applicable',
    assumptions='Resampling with replacement; no distributional assumption.',
    statistic=f'median = {np.median(x):.4f}',
    df='-',
    p_value=np.nan,
    interpretation=f'Bootstrap 95% CI for median HFVS is ({boot_ci[0]:.3f}, {boot_ci[1]:.3f}).'
)

print(f'Bootstrap median CI: ({boot_ci[0]:.4f}, {boot_ci[1]:.4f})')


## 10. Construct validation and categorical association tests

The business document emphasizes validation: constructed financial stress should align with stated burden and missed payments. These chi-square tests check whether key binary constructed indicators are associated with respondent-reported or risk outcomes.


In [ ]:
# 10.1 Chi-square validation tests

def chi_square_test(row_var, col_var, label):
    sub = analysis[[row_var, col_var]].dropna().copy()
    if sub.empty:
        print(f'Skipping {label}: no complete rows.')
        return None
    table = pd.crosstab(sub[row_var], sub[col_var])
    if table.shape[0] < 2 or table.shape[1] < 2:
        print(f'Skipping {label}: table is not at least 2x2.')
        return None
    chi2, p, dof, expected = stats.chi2_contingency(table)
    v = cramers_v(table)
    add_result(
        test='Chi-square test of independence',
        variables=f'{row_var} x {col_var}',
        question=label,
        h0=f'{row_var} and {col_var} are independent',
        h1=f'{row_var} and {col_var} are associated',
        assumptions='Categorical variables; expected cell counts reviewed by chi-square procedure.',
        statistic=f'chi2 = {chi2:.4f}; Cramers V = {v:.4f}',
        df=f'{dof}',
        p_value=p,
        interpretation=f'Cramers V={v:.3f}; larger values imply stronger categorical association.'
    )
    print('\n' + label)
    print(table)
    print(f'chi2={chi2:.4f}, dof={dof}, p={p:.6g}, Cramers V={v:.3f}')
    return table

validation_tables = {}
validation_tables['rent_vs_stated_burden'] = chi_square_test(
    'rent_burdened_model', 'housing_cost_burden',
    'Does constructed rent burden align with stated housing-cost burden?'
)
validation_tables['rent_vs_missed_payment'] = chi_square_test(
    'rent_burdened_model', 'missed_payment',
    'Does constructed rent burden align with missed housing payments?'
)
validation_tables['triple_vs_eviction'] = chi_square_test(
    'triple_exposed', 'eviction_risk',
    'Are triple-exposed households more likely to report eviction risk?'
)
validation_tables['overcrowded_vs_perception'] = chi_square_test(
    'is_overcrowded', 'perc_overall',
    'Is overcrowding associated with subjective overall housing perception?'
)


## 11. County-level risk atlas

The business understanding document requires survey weights for county-level outputs. This section creates a policy-facing county atlas with weighted mean HFVS, triple exposure, overcrowding, rent burden, and mortgage-market exclusion.


In [ ]:
# 11.1 Weighted county summary
county_rows = []
group_cols = ['county_code', 'county_name'] if 'county_name' in analysis.columns else ['county_code']

for keys, g in analysis.groupby(group_cols, dropna=False):
    if not isinstance(keys, tuple):
        keys = (keys,)
    row = dict(zip(group_cols, keys))
    row.update({
        'n_households': len(g),
        'weighted_mean_hfvs': weighted_mean(g['hfvs_score'], g['hh_weight_clean']),
        'unweighted_mean_hfvs': g['hfvs_score'].mean(),
        'median_hfvs': g['hfvs_score'].median(),
        'pct_high_or_very_high': g['hfvs_class'].isin(['High', 'Very high']).mean(),
        'pct_triple_exposed': binary01('triple_exposed', g).mean(),
        'pct_overcrowded': binary01('is_overcrowded', g).mean(),
        'pct_rent_burdened_renters': g['rent_burdened_model'].mean(),
        'pct_mortgage_market_absent': binary01('mort_no_market', g).mean(),
        'median_total_exp': g['total_exp'].median(),
    })
    county_rows.append(row)

county_atlas = pd.DataFrame(county_rows)
county_atlas['hfvs_rank'] = county_atlas['weighted_mean_hfvs'].rank(ascending=False, method='min').astype(int)
county_atlas = county_atlas.sort_values('weighted_mean_hfvs', ascending=False)
county_atlas.to_csv(TABS / 'statistical_inference_county_risk_atlas.csv', index=False)

display(county_atlas.head(15))


In [ ]:
# 11.2 County visual summaries
top = county_atlas.head(15).copy()
label_col = 'county_name' if 'county_name' in top.columns else 'county_code'

fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(data=top, y=label_col, x='weighted_mean_hfvs', color='#B85C38', ax=ax)
ax.set_title('Top 15 Counties by Weighted Mean HFVS')
ax.set_xlabel('Weighted mean HFVS')
ax.set_ylabel('County')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_top_counties_hfvs.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(
    data=county_atlas,
    x='pct_triple_exposed',
    y='weighted_mean_hfvs',
    size='n_households',
    hue='pct_mortgage_market_absent',
    palette='viridis',
    sizes=(40, 400),
    ax=ax,
)
ax.set_title('County Vulnerability: Triple Exposure and Mortgage-Market Absence')
ax.set_xlabel('Triple-exposure rate')
ax.set_ylabel('Weighted mean HFVS')
fig.tight_layout()
fig.savefig(FIGS / 'statistical_inference_county_triple_vs_hfvs.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Consolidated results table

The table below is the report-ready statistical inference table requested in the workflow. It contains parametric tests, nonparametric tests, validation tests, and the regression model in one consistent structure.


In [ ]:
# 12.1 Save and display all inference results
results = pd.DataFrame(TEST_RESULTS)
if not results.empty:
    results['p_value_fmt'] = results['p_value'].map(lambda p: 'NA' if pd.isna(p) else f'{p:.4g}')
    results.to_csv(TABS / 'statistical_inference_test_results.csv', index=False)
    display_cols = ['test', 'variables', 'statistic', 'df', 'p_value_fmt', 'decision_alpha_0_05', 'plain_english_interpretation']
    display(results[display_cols])
    print(f'Saved test results -> {TABS / "statistical_inference_test_results.csv"}')
else:
    print('No test results were generated.')


In [ ]:
# 12.2 Final export checklist
exports = pd.DataFrame({
    'artifact': [
        'Reduced <=50 source-column frame',
        'Scored analysis frame with HFVS derived variables',
        'Column reduction audit',
        'Descriptive statistics',
        'Normality results',
        'Inference test results',
        'Regression coefficients',
        'County risk atlas',
        'Figures folder',
    ],
    'path': [
        str(PQ / 'statistical_inference_analysis_frame_50cols.parquet'),
        str(PQ / 'statistical_inference_scored_frame.parquet'),
        str(TABS / 'statistical_inference_column_reduction_audit.csv'),
        str(TABS / 'statistical_inference_descriptive_statistics.csv'),
        str(TABS / 'statistical_inference_normality_results.csv'),
        str(TABS / 'statistical_inference_test_results.csv'),
        str(TABS / 'statistical_inference_regression_coefficients.csv'),
        str(TABS / 'statistical_inference_county_risk_atlas.csv'),
        str(FIGS),
    ]
})

display(exports)
print('Notebook complete. Re-run all cells before report export so outputs and saved artifacts are current.')


## Report writing notes

Use these results to structure the final report:

- **Introduction:** housing financial vulnerability in Kenya; why affordability, quality, tenure, hazards, and utilities must be analyzed together.
- **Data and preprocessing:** cite the 21,347-household KHS master frame, the reduction from 431 cleaned columns to <=50 inference columns, and the structural-missingness logic for renters, land ownership, and mortgage-market absence.
- **EDA:** include HFVS/pillar distributions, the correlation heatmap, the expenditure scatter plot, and objective-subjective quality gap findings.
- **Methods:** explain normality screening and why both parametric and nonparametric tests are reported.
- **Results:** use `statistical_inference_test_results.csv` as the master table.
- **Discussion:** compare parametric and nonparametric conclusions, highlight county-level risk patterns, and identify where constructed indicators align or fail to align with self-reported burden.
- **Limitations:** cross-sectional data, codebook assumptions for some ordinal categories, formula-constructed HFVS, and association rather than causation.
